In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [14]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [15]:
len(docs)

9

In [16]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_docs = splitter.split_documents(docs)

In [17]:
len(splitter_docs)

26

In [18]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = InMemoryVectorStore.from_documents(
    documents = splitter_docs,
    embedding=embeddings
)

In [19]:
same_record = vector_store.similarity_search("Patient name")

In [20]:
### agent = tools, llm, prompt

In [ ]:
@tool

def retriever_tool(query:str):
    """
       This  tools can help youto retriever the relevant data of the PDF Documents, and these pdf
       documents have details about medical reports. 
    """
    docs = vector_store.similarity_search(query=query, k=4)
    context =""

    for doc in docs:
        context = doc.page_content + "\n\n"

    return context 

In [29]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [30]:
System_Prompt = """
    You are a helpful assiatant that answers questions using retrieved context.
    ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [33]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [34]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

In [37]:
result = response["messages"][-1].text

In [38]:
print(result)

The patient's name is Ms. NIKITA CHUDHARY and the doctor's name is DR NITIN NAHAR.
